In [0]:
import boto3
import csv
import json
import io
import random
from datetime import datetime, timezone, date


### define env vars

In [0]:
# Accessing data directly through Spark/DBFS
BUCKET = 's3://fraud-demo-bucket-043309328060-us-east-1-an/'

### define exps 

In [0]:

MCC_CATEGORIES = [
    "Airlines", "Auto Dealers", "Clothing Stores",
    "Grocery Stores", "Restaurants", "Electronics",
    "Government Services", "Healthcare", "Hotels",
    "Insurance", "Online Retail", "Utilities",
]
RISK_LEVELS = ["LOW","LOW","LOW","MEDIUM","MEDIUM","HIGH"]
COUNTRIES   = ["GB","US","EU","DE","FR","JP","CN","AU","CA"]
ENTITY_TYPES = ["INDIVIDUAL","ORGANISATION","VESSEL","AIRCRAFT"]
SANCTIONS_PROGRAMS = [
    "OFAC_SDN","UN_CONSOLIDATED","EU_CONSOLIDATED",
    "HMT_CONSOLIDATED","OFAC_CONSOLIDATED"
]


### define functions

In [0]:

def generate_mcc_csv(n=500):
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=[
        "mcc_code","description","category",
        "risk_level","requires_enhanced_dd",
        "effective_date","last_updated"
    ])
    writer.writeheader()
    for i in range(n):
        writer.writerow({
            "mcc_code":           str(1000 + i).zfill(4),
            "description":        f"Merchant type {1000+i}",
            "category":           random.choice(MCC_CATEGORIES),
            "risk_level":         random.choice(RISK_LEVELS),
            "requires_enhanced_dd": random.choice(["Y","N","N","N"]),
            "effective_date":     "2024-01-01",
            "last_updated":       date.today().isoformat(),
        })
    return buf.getvalue().encode("utf-8")

def generate_sanctions_json(n=200):
    records = []
    for i in range(n):
        records.append({
            "entity_id":        f"ENT{str(i+1).zfill(8)}",
            "entity_type":      random.choice(ENTITY_TYPES),
            "full_name":        f"Sanctioned Entity {i+1}",
            "aliases":          [f"Alias {i+1}A", f"Alias {i+1}B"],
            "country_of_origin":random.choice(COUNTRIES),
            "sanctions_program":random.choice(SANCTIONS_PROGRAMS),
            "listed_date":      "2023-06-15",
            "last_updated":     date.today().isoformat(),
            "is_active":        random.choice([True,True,True,False]),
            "risk_score":       round(random.uniform(0.5, 1.0), 3),
            "source_list":      random.choice(SANCTIONS_PROGRAMS),
        })
    return json.dumps({
        "schema_version": "1.0",
        "generated_at":   datetime.now(timezone.utc).isoformat(),
        "record_count":   n,
        "records":        records
    }, indent=2).encode("utf-8")


In [0]:

# Assuming 'df' is your Spark DataFrame
df = spark.createDataFrame(generate_mcc_csv())

# Writing as CSV
df.write \
.mode("overwrite") \
.option("header", "true") \
.csv("s3://fraud-demo-bucket-043309328060-us-east-1-an/reference/mcc_codes")


df = spark.createDataFrame(generate_sanctions_json())

# Writing as JSON
df.write \
.mode("overwrite") \
.json("s3://fraud-demo-bucket-043309328060-us-east-1-an/reference/sanctions")


print("Done.")